In [1]:
! pip install transformers datasets torch scikit-learn

In [2]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

# LOADING THE ACTUAL DATASET

In [3]:
dataset = load_dataset('stanfordnlp/imdb')
dataset = dataset.shuffle(seed=42)
small_train = dataset['train'].select(range(500))
small_test = dataset['test'].select(range(200))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenizer_function(example):
  return tokenizer(example['text'],padding='max_length',truncation=True)

train_enc = small_train.map(tokenizer_function,batched=True)
test_enc = small_test.map(tokenizer_function,batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [6]:
train_enc

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

# Creating Tensors

In [7]:
train_enc.set_format('torch',columns=['input_ids','attention_mask','label'])
test_enc.set_format('torch',columns=['input_ids','attention_mask','label'])

In [8]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Evaluation Function

In [9]:
def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  labels = labels.astype('int')
  preds = preds.astype('int')
  precision, recall, f1, _ = precision_recall_fscore_support(labels,preds,average='binary')
  acc = accuracy_score(labels,preds)
  return {
      'accuracy':float(acc),
      'f1':float(f1),
      'precision':float(precision),
      'recall':float(recall)
  }

# Setting up Training Config

In [15]:
training_args = TrainingArguments(
    output_dir='.results',
    eval_strategy='epoch',
    learning_rate = 2e-5,
    num_train_epochs=2,
    optim='adamw_torch_xla'
)

In [16]:
trainer =Trainer(
    model = model,
    args = training_args,
    train_dataset = train_enc,
    eval_dataset = test_enc,
    compute_metrics = compute_metrics
)

In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.574651,0.750000,0.778761,0.676923,0.916667
2,No log,0.414398,0.830000,0.839623,0.767241,0.927083


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=126, training_loss=0.5034693763369605, metrics={'train_runtime': 81.3117, 'train_samples_per_second': 12.397, 'train_steps_per_second': 1.55, 'total_flos': 263111055360000.0, 'train_loss': 0.5034693763369605, 'epoch': 2.0})

In [18]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.4143978953361511,
 'eval_accuracy': 0.83,
 'eval_f1': 0.839622641509434,
 'eval_precision': 0.7672413793103449,
 'eval_recall': 0.9270833333333334,
 'eval_runtime': 0.715,
 'eval_samples_per_second': 279.723,
 'eval_steps_per_second': 34.965,
 'epoch': 2.0}

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

inputs = tokenizer("Some input", return_tensors="pt").to(device)

outputs = model(**inputs)

In [31]:
text = "this is a good movie and very emotional"

prediction = torch.argmax(outputs.logits).item()

print("Predicted Sentiments:","Positive" if prediction == 1 else "Negative")

Predicted Sentiments: Negative
